# Train Parts A–C on Colab (A100)

Runtime → Change runtime type → **A100 GPU**. Then run the cells top to bottom.

Checkpoints are written straight to Google Drive (through symlinks), and `results/` is copied to Drive after every part, so a disconnect loses at most one epoch. Weights stay out of git: upload them from Drive to Hugging Face or share the Drive folder, and link them in the README.

Rough times on an A100: MAE ~10–20 min, classifier ~1 min, VAE ~2 min, DDPM ~20–30 min plus ~2 min of sampling.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/Areesha-008/generative-models.git"
BRANCH = "main"   # before the PRs are merged, use "feat/app" (it has every part)
REPO = "/content/generative-models"
DRIVE_DIR = "/content/drive/MyDrive/generative-models-runs"
UHAT_URL = "https://zenodo.org/records/3670611/files/data.zip?download=1"

## 2. Drive, code, packages, data

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DRIVE_DIR, exist_ok=True)

In [ ]:
import os
if os.path.isdir(REPO):
    !git -C {REPO} fetch -q origin && git -C {REPO} checkout -q {BRANCH} && git -C {REPO} pull -q origin {BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} {REPO}
%cd {REPO}
!git log --oneline -3

In [ ]:
# Colab already has torch, transformers, datasets and matplotlib; this adds diffusers and rouge-score.
!pip install -q -r requirements.txt
import torch, diffusers
print("torch", torch.__version__, "| diffusers", diffusers.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
# UHaT (32.9 MB) from Zenodo, unzipped into starter/data/uhat/
import os
if not os.path.isdir("starter/data/uhat/data/characters_train_set"):
    !wget -q -O /content/uhat.zip "{UHAT_URL}"
    !unzip -q -o /content/uhat.zip -d starter/data/uhat/
!ls starter/data/uhat/data

In [ ]:
# Checkpoint folders live on Drive; each part writes to part_x/checkpoints/ as usual.
import os
for part in ["part_a_mae", "part_b_vae", "part_c_diffusion"]:
    target = f"{DRIVE_DIR}/checkpoints/{part}"
    os.makedirs(target, exist_ok=True)
    link = f"{REPO}/{part}/checkpoints"
    if not os.path.islink(link):
        if os.path.isdir(link):
            os.rmdir(link)
        os.symlink(target, link)
!ls -l part_*/ | grep checkpoints

import shutil
def sync():
    """Copy results/ and starter/results/ to Drive."""
    shutil.copytree(f"{REPO}/results", f"{DRIVE_DIR}/results", dirs_exist_ok=True)
    shutil.copytree(f"{REPO}/starter/results", f"{DRIVE_DIR}/starter_results", dirs_exist_ok=True)
    print("synced to", DRIVE_DIR)

## 3. Task 0 – starter check
The printed shapes and counts go in the README; the two grids land in `starter/results/`.

In [ ]:
%cd {REPO}/starter
!python check_starter.py 2>&1 | tee results/check_starter_output.txt
%cd {REPO}
sync()

## 4. Correctness checks (§3.6)
The outputs are saved to `results/checks_*.txt` for the README.

In [ ]:
!python part_a_mae/model.py 2>&1 | tee results/checks_part_a.txt
!python part_b_vae/model.py 2>&1 | tee results/checks_part_b.txt
!python part_c_diffusion/diffusion.py 2>&1 | tee results/checks_part_c.txt

## 5. Part A – MAE pretraining (100 epochs) and few-label classifier

In [ ]:
!python part_a_mae/pretrain.py 2>&1 | tee results/part_a_pretrain_log.txt
sync()

In [ ]:
!python part_a_mae/classify.py 2>&1 | tee results/part_a_classify_log.txt
sync()

## 6. Part B – VAE (30 epochs)

In [ ]:
!python part_b_vae/train.py 2>&1 | tee results/part_b_log.txt
sync()

## 7. Part C – DDPM (40 epochs) and sampling

In [ ]:
!python part_c_diffusion/train.py 2>&1 | tee results/part_c_log.txt
sync()

### Part C blind test (§3.3)
One shuffled sample per character. Each member fills in their own `results/blind_member*.csv` without seeing the other's sheet or `results/part_c_blind_key.csv`, then runs `python part_c_diffusion/blind_test.py score` (on a laptop is fine).

In [ ]:
!python part_c_diffusion/blind_test.py generate
sync()
from IPython.display import Image, display
display(Image("results/part_c_blind_test.png", width=700))

## 8. Look at the figures

In [ ]:
from IPython.display import Image, display
import glob
for path in sorted(glob.glob("results/*.png")) + sorted(glob.glob("starter/results/*.png")):
    print(path)
    display(Image(path, width=700))

In [ ]:
# Table 2 inputs and headline numbers
!for f in results/part_*_training.txt results/part_a_test_accuracy.txt; do echo "== $$f"; cat $$f; done

## 9. Take the results home

`results/` and `starter/results/` are on Drive under `generative-models-runs/`. Copy them into your local clone and commit from your own machine so the commits carry your name. Checkpoints are in `generative-models-runs/checkpoints/`: upload them to Hugging Face or share the folder, and link them in the README. Do not commit them.

In [ ]:
!cd {REPO} && zip -qr /content/results.zip results starter/results && cp /content/results.zip {DRIVE_DIR}/
print("results.zip is in", DRIVE_DIR)